In [1]:
# Imports
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import hashlib
import json

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
METADATA_PATH = DATA_DIR / "HAM10000_metadata.csv"
IMAGE_PART1 = DATA_DIR / "HAM10000_images_part_1"
IMAGE_PART2 = DATA_DIR / "HAM10000_images_part_2"

# Create validation output dir
VAL_DIR = PROJECT_ROOT / "data" / "validation_reports"
VAL_DIR.mkdir(parents=True, exist_ok=True)

print("Data Validation Pipeline Started")

Data Validation Pipeline Started


In [ ]:
# Load metadata
df = pd.read_csv(METADATA_PATH)

print("=== METADATA VALIDATION ===\n")

# Basic checks
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Check required columns exist
required_cols = ['lesion_id', 'image_id', 'dx', 'age', 'sex', 'localization']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    print(f"✗ Missing columns: {missing_cols}")
else:
    print("✓ All required columns present")

# Check data types
print("\nData types:")
print(df.dtypes)

# Check for empty dataframe
if len(df) == 0:
    print("✗ ERROR: Empty dataframe!")
else:
    print(f"✓ Loaded {len(df)} records")

=== METADATA VALIDATION ===

Shape: (10015, 7)
Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']
✓ All required columns present

Data types:
lesion_id           str
image_id            str
dx                  str
dx_type             str
age             float64
sex                 str
localization        str
dtype: object
✓ Loaded 10015 records


In [3]:
print("=== LABEL VALIDATION ===\n")

# Expected 7 classes
expected_classes = {'akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'}
actual_classes = set(df['dx'].unique())

print(f"Expected classes: {sorted(expected_classes)}")
print(f"Actual classes: {sorted(actual_classes)}")

# Check for unexpected classes
unexpected = actual_classes - expected_classes
if unexpected:
    print(f"✗ Unexpected classes found: {unexpected}")
else:
    print("✓ All classes are valid")

# Check for missing classes
missing = expected_classes - actual_classes
if missing:
    print(f"✗ Missing classes: {missing}")
else:
    print("✓ All expected classes present")

# Check for null labels
null_labels = df['dx'].isnull().sum()
if null_labels > 0:
    print(f"✗ {null_labels} rows with null labels")
else:
    print("✓ No null labels")

# Class distribution
print("\nClass distribution:")
print(df['dx'].value_counts())

=== LABEL VALIDATION ===

Expected classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
Actual classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
✓ All classes are valid
✓ All expected classes present
✓ No null labels

Class distribution:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


In [4]:
print("=== ID VALIDATION ===\n")

# Check for null IDs
null_lesion = df['lesion_id'].isnull().sum()
null_image = df['image_id'].isnull().sum()

print(f"Null lesion_ids: {null_lesion}")
print(f"Null image_ids: {null_image}")

if null_lesion == 0 and null_image == 0:
    print("✓ No null IDs")
else:
    print("✗ Found null IDs!")

# Check for duplicate image_ids (should be unique)
duplicate_images = df['image_id'].duplicated().sum()
print(f"\nDuplicate image_ids: {duplicate_images}")

if duplicate_images == 0:
    print("✓ All image_ids are unique")
else:
    print("✗ Found duplicate image_ids!")
    print(df[df['image_id'].duplicated(keep=False)].head())

# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicate_rows}")

if duplicate_rows == 0:
    print("✓ No duplicate rows")
else:
    print("✗ Found duplicate rows!")

# Check lesion_id consistency
print(f"\nUnique lesion_ids: {df['lesion_id'].nunique()}")
print(f"Unique image_ids: {df['image_id'].nunique()}")
print(f"Images per lesion stats:")
lesion_counts = df.groupby('lesion_id').size()
print(lesion_counts.describe())

=== ID VALIDATION ===

Null lesion_ids: 0
Null image_ids: 0
✓ No null IDs

Duplicate image_ids: 0
✓ All image_ids are unique

Duplicate rows: 0
✓ No duplicate rows

Unique lesion_ids: 7470
Unique image_ids: 10015
Images per lesion stats:
count    7470.000000
mean        1.340696
std         0.633970
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         6.000000
dtype: float64


In [5]:
print("=== IMAGE FILE VALIDATION ===\n")

# Check directories exist
print(f"Image part 1 exists: {IMAGE_PART1.exists()}")
print(f"Image part 2 exists: {IMAGE_PART2.exists()}")

if not IMAGE_PART1.exists() or not IMAGE_PART2.exists():
    raise FileNotFoundError("Image directories not found!")

# Get all image files
part1_files = set(f.stem for f in IMAGE_PART1.glob("*.jpg"))
part2_files = set(f.stem for f in IMAGE_PART2.glob("*.jpg"))
all_image_files = part1_files | part2_files

print(f"\nImages in part 1: {len(part1_files)}")
print(f"Images in part 2: {len(part2_files)}")
print(f"Total unique images on disk: {len(all_image_files)}")

# Check for overlap
overlap = part1_files & part2_files
print(f"Images in both folders: {len(overlap)}")

if len(overlap) > 0:
    print(f"✗ WARNING: {len(overlap)} images duplicated across folders")
    print(f"  Examples: {list(overlap)[:5]}")
else:
    print("✓ No duplicate images across folders")

# Compare metadata with disk
metadata_images = set(df['image_id'])
missing_on_disk = metadata_images - all_image_files
extra_on_disk = all_image_files - metadata_images

print(f"\nImages in metadata: {len(metadata_images)}")
print(f"Images missing from disk: {len(missing_on_disk)}")
print(f"Extra images on disk: {len(extra_on_disk)}")

if len(missing_on_disk) == 0:
    print("✓ All metadata images exist on disk")
else:
    print(f"✗ {len(missing_on_disk)} images missing!")
    print(f"  Examples: {list(missing_on_disk)[:10]}")

if len(extra_on_disk) > 0:
    print(f"✓ {len(extra_on_disk)} extra images on disk (not in metadata)")

=== IMAGE FILE VALIDATION ===

Image part 1 exists: True
Image part 2 exists: True

Images in part 1: 5000
Images in part 2: 5015
Total unique images on disk: 10015
Images in both folders: 0
✓ No duplicate images across folders

Images in metadata: 10015
Images missing from disk: 0
Extra images on disk: 0
✓ All metadata images exist on disk


In [6]:
print("=== IMAGE INTEGRITY CHECK ===\n")

def get_image_path(image_id):
    """Find image in either part_1 or part_2"""
    img_name = f"{image_id}.jpg"
    path1 = IMAGE_PART1 / img_name
    path2 = IMAGE_PART2 / img_name
    if path1.exists():
        return path1
    elif path2.exists():
        return path2
    return None

def check_image_integrity(image_id):
    """Check if image can be opened and has valid properties"""
    path = get_image_path(image_id)
    if path is None:
        return False, "File not found", None
    
    try:
        with Image.open(path) as img:
            # Check image mode
            if img.mode not in ['RGB', 'L', 'RGBA']:
                return False, f"Invalid mode: {img.mode}", None
            
            # Check size
            w, h = img.size
            if w == 0 or h == 0:
                return False, "Zero dimension", None
            
            # Try to load pixel data
            img.load()
            
            return True, "OK", {"width": w, "height": h, "mode": img.mode}
    except Exception as e:
        return False, str(e), None

# Sample check (full check takes time - check all or sample)
CHECK_SAMPLE = True  # Set to False for full validation
SAMPLE_SIZE = 100

if CHECK_SAMPLE:
    print(f"Checking sample of {SAMPLE_SIZE} images...")
    sample_ids = df['image_id'].sample(SAMPLE_SIZE, random_state=42).tolist()
else:
    print(f"Checking all {len(df)} images...")
    sample_ids = df['image_id'].tolist()

# Check images
corrupted = []
valid_count = 0
image_stats = []

for img_id in sample_ids:
    is_valid, msg, stats = check_image_integrity(img_id)
    if is_valid:
        valid_count += 1
        image_stats.append(stats)
    else:
        corrupted.append({"image_id": img_id, "error": msg})

print(f"\nChecked: {len(sample_ids)}")
print(f"Valid: {valid_count}")
print(f"Corrupted: {len(corrupted)}")

if len(corrupted) > 0:
    print(f"\n✗ Found {len(corrupted)} corrupted images:")
    for c in corrupted[:5]:
        print(f"  {c['image_id']}: {c['error']}")
else:
    print("✓ All checked images are valid")

# Analyze image statistics
if image_stats:
    widths = [s['width'] for s in image_stats]
    heights = [s['height'] for s in image_stats]
    modes = [s['mode'] for s in image_stats]
    
    print(f"\nImage statistics (sample):")
    print(f"  Width: {min(widths)} - {max(widths)} (mean: {np.mean(widths):.0f})")
    print(f"  Height: {min(heights)} - {max(heights)} (mean: {np.mean(heights):.0f})")
    print(f"  Modes: {set(modes)}")

=== IMAGE INTEGRITY CHECK ===

Checking sample of 100 images...

Checked: 100
Valid: 100
Corrupted: 0
✓ All checked images are valid

Image statistics (sample):
  Width: 600 - 600 (mean: 600)
  Height: 450 - 450 (mean: 450)
  Modes: {'RGB'}


In [7]:
print("=== METADATA CONSISTENCY ===\n")

# Check lesion_id has consistent dx across all its images
print("Checking lesion consistency...")

lesion_class_check = df.groupby('lesion_id')['dx'].nunique()
inconsistent_lesions = lesion_class_check[lesion_class_check > 1]

print(f"Lesions with multiple class labels: {len(inconsistent_lesions)}")

if len(inconsistent_lesions) > 0:
    print("✗ Found inconsistent lesions!")
    for lesion_id in inconsistent_lesions.head(3).index:
        classes = df[df['lesion_id'] == lesion_id]['dx'].unique()
        print(f"  {lesion_id}: {classes}")
else:
    print("✓ All lesions have consistent class labels")

# Check for suspicious values
print(f"\nAge range: {df['age'].min()} - {df['age'].max()}")
ages_above_100 = (df['age'] > 100).sum()
if ages_above_100 > 0:
    print(f"⚠ {ages_above_100} patients with age > 100")
else:
    print("✓ Age values look reasonable")

# Check sex values
print(f"\nSex values: {df['sex'].unique()}")
unexpected_sex = set(df['sex'].unique()) - {'male', 'female', 'unknown'}
if unexpected_sex:
    print(f"⚠ Unexpected sex values: {unexpected_sex}")
else:
    print("✓ Sex values are valid")

# Check localization
print(f"\nNumber of unique locations: {df['localization'].nunique()}")
print(f"Top locations:\n{df['localization'].value_counts().head()}")

=== METADATA CONSISTENCY ===

Checking lesion consistency...
Lesions with multiple class labels: 0
✓ All lesions have consistent class labels

Age range: 0.0 - 85.0
✓ Age values look reasonable

Sex values: <StringArray>
['male', 'female', 'unknown']
Length: 3, dtype: str
✓ Sex values are valid

Number of unique locations: 15
Top locations:
localization
back               2192
lower extremity    2077
trunk              1404
upper extremity    1118
abdomen            1022
Name: count, dtype: int64


In [8]:
print("=== GENERATING VALIDATION REPORT ===\n")

# Compile validation results
validation_report = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "total_images": len(df),
    "total_lesions": df['lesion_id'].nunique(),
    
    "metadata_checks": {
        "required_columns_present": len(missing_cols) == 0,
        "missing_columns": missing_cols,
        "null_labels": int(null_labels),
        "duplicate_images": int(duplicate_images),
        "duplicate_rows": int(duplicate_rows)
    },
    
    "label_checks": {
        "expected_classes": sorted(expected_classes),
        "actual_classes": sorted(actual_classes),
        "unexpected_classes": sorted(unexpected),
        "missing_classes": sorted(missing),
        "all_classes_valid": len(unexpected) == 0 and len(missing) == 0
    },
    
    "id_checks": {
        "null_lesion_ids": int(null_lesion),
        "null_image_ids": int(null_image),
        "unique_lesion_ids": int(df['lesion_id'].nunique()),
        "unique_image_ids": int(df['image_id'].nunique())
    },
    
    "file_checks": {
        "images_in_part1": len(part1_files),
        "images_in_part2": len(part2_files),
        "images_in_metadata": len(metadata_images),
        "missing_on_disk": len(missing_on_disk),
        "extra_on_disk": len(extra_on_disk),
        "all_files_present": len(missing_on_disk) == 0
    },
    
    "integrity_checks": {
        "sample_checked": len(sample_ids),
        "valid_images": valid_count,
        "corrupted_images": len(corrupted),
        "corruption_list": corrupted[:10]  # First 10 only
    },
    
    "consistency_checks": {
        "inconsistent_lesions": len(inconsistent_lesions),
        "all_consistent": len(inconsistent_lesions) == 0
    },
    
    "overall_status": "PASS" if (
        len(missing_cols) == 0 and
        len(unexpected) == 0 and
        len(missing) == 0 and
        len(missing_on_disk) == 0 and
        len(corrupted) == 0 and
        len(inconsistent_lesions) == 0
    ) else "FAIL"
}

# Save report
report_path = VAL_DIR / "validation_report.json"
with open(report_path, "w") as f:
    json.dump(validation_report, f, indent=2)

print(f"✓ Report saved to {report_path}")
print(f"\nOverall status: {validation_report['overall_status']}")

# Print summary
print("\n" + "=" * 50)
print("VALIDATION SUMMARY")
print("=" * 50)

checks = [
    ("Required columns", validation_report['metadata_checks']['required_columns_present']),
    ("Valid classes", validation_report['label_checks']['all_classes_valid']),
    ("No null IDs", validation_report['id_checks']['null_lesion_ids'] == 0),
    ("All files present", validation_report['file_checks']['all_files_present']),
    ("No corruption", len(corrupted) == 0),
    ("Consistent lesions", validation_report['consistency_checks']['all_consistent'])
]

for check_name, passed in checks:
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"{status}: {check_name}")

print("=" * 50)

=== GENERATING VALIDATION REPORT ===

✓ Report saved to d:\skin_screening_system\data\validation_reports\validation_report.json

Overall status: PASS

VALIDATION SUMMARY
✓ PASS: Required columns
✓ PASS: Valid classes
✓ PASS: No null IDs
✓ PASS: All files present
✓ PASS: No corruption
✓ PASS: Consistent lesions


In [9]:
if validation_report['overall_status'] == "PASS":
    print("=== SAVING CLEAN METADATA ===\n")
    
    # Add image path column
    def get_image_path_str(image_id):
        img_name = f"{image_id}.jpg"
        if (IMAGE_PART1 / img_name).exists():
            return str(IMAGE_PART1 / img_name)
        elif (IMAGE_PART2 / img_name).exists():
            return str(IMAGE_PART2 / img_name)
        return None
    
    df['image_path'] = df['image_id'].apply(get_image_path_str)
    
    # Save clean version
    clean_path = DATA_DIR / "HAM10000_metadata_clean.csv"
    df.to_csv(clean_path, index=False)
    print(f"✓ Clean metadata saved to {clean_path}")
    print(f"  Added column: image_path")
    print(f"\nReady for Stage 3: Data Splitting")
else:
    print("✗ VALIDATION FAILED - Fix issues before proceeding!")
    print("\nCritical issues to fix:")
    if validation_report['metadata_checks']['missing_columns']:
        print(f"  - Missing columns: {validation_report['metadata_checks']['missing_columns']}")
    if not validation_report['label_checks']['all_classes_valid']:
        print("  - Invalid class labels")
    if validation_report['file_checks']['missing_on_disk'] > 0:
        print(f"  - {validation_report['file_checks']['missing_on_disk']} images missing")
    if len(corrupted) > 0:
        print(f"  - {len(corrupted)} corrupted images")
    if not validation_report['consistency_checks']['all_consistent']:
        print("  - Inconsistent lesion labels")

=== SAVING CLEAN METADATA ===

✓ Clean metadata saved to d:\skin_screening_system\data\raw\HAM10000_metadata_clean.csv
  Added column: image_path

Ready for Stage 3: Data Splitting
